In [39]:
import numpy as np
from numpy import where
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from collections import Counter
warnings.filterwarnings('ignore')
import matplotlib.ticker as mtick
import matplotlib.gridspec as grid_spec
from matplotlib.offsetbox import AnchoredText
from mpl_toolkits.axes_grid1 import make_axes_locatable


# Data pre-processing 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier

# Modelling 
from sklearn.metrics import (
    roc_auc_score,
    precision_score, 
    accuracy_score, 
    recall_score,
)


from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    AdaBoostClassifier, 
    GradientBoostingClassifier)



from feature_engine import encoding as ce


from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score, cross_validate, RepeatedStratifiedKFold



from sklearn.pipeline import Pipeline

from imblearn.over_sampling import ADASYN, SMOTE

from imblearn.pipeline import make_pipeline
from imblearn.pipeline import Pipeline as imbPipeline


import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np 
import pandas as pd

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline as imbPipeline
from feature_engine import encoding as ce
from sklearn.utils.class_weight import compute_sample_weight


In [40]:
df=pd.read_csv('train_apps.csv',sep=',')

In [41]:
df.head()

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,...,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea,target_value
0,127345,2024-02-01,1.339991,-1.847954,-1.586546,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lombard,NaN,0
1,127209,2024-02-01,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,0.842771,NaN,NaN,...,NaN,NaN,NaN,-0.862289,-3.400318,-0.780786,NaN,inn_scoring,NaN,0
2,272776,2024-02-01,2.185431,3.167063,2.369547,-0.709770,-0.400695,0.000000,0.834373,4.897257,...,NaN,NaN,NaN,1.168810,3.015012,0.554064,NaN,NaN,NaN,0
3,127210,2024-02-01,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,164500,2024-02-01,0.845440,4.559196,3.467730,-2.484194,-0.400695,0.000000,-0.518122,-3.435251,...,NaN,1.067447,NaN,1.022327,1.506380,0.190096,NaN,NaN,NaN,0


In [42]:
df.isnull().sum()

front_id                                  0
decision_day                              0
loan_amount_last                          0
overdraft_limit_min                       0
overdraft_limit_max                       0
offered_rate                              0
cb_rate                                   0
corp_credit_products                  51188
sum_deb_ul_90                         54111
sum_deb_ul_30                         61453
cnt_deb_loan_90                       31458
cnt_deb_ul_ip_90                      30305
cnt_deb_ul_ip_30                      33373
balance_rur_amt_30_min                34826
cnt_cred_loan_90                      31458
loan_rev_max_start_non_fin           132635
loan_rev_min_start_fin               124706
app_term_mean_360                     55883
overdraft_app_term_max_360           139732
days_from_authperson_registration     78473
fl_hdb_bki_total_active_products      24368
corp_list                             51188
count_all_corp_dashboard_events 

In [43]:
df['rate_spread'] = df['offered_rate'] - df['cb_rate']
df['rate_ratio'] = df['offered_rate'] / df['cb_rate']

In [44]:
df['log_rate_spread'] = np.log1p(df['rate_spread'].clip(lower=0))

In [45]:
df['od_limit_avg'] = (df['overdraft_limit_min'] + df['overdraft_limit_max']) / 2
df['loan_to_od_max'] = df['loan_amount_last'] / df['overdraft_limit_max']
df['loan_to_od_avg'] = df['loan_amount_last'] / df['od_limit_avg']


In [46]:
df['loan_to_od_max_is_inf'] = np.isinf(df['loan_to_od_max']).astype(int)
median_val = df.loc[~np.isinf(df['loan_to_od_max']), 'loan_to_od_max'].median()
df['loan_to_od_max'].replace([np.inf, -np.inf], median_val, inplace=True)
df['loan_to_od_max'].fillna(median_val, inplace=True)
df['loan_to_od_avg'].isnull().sum()

np.int64(0)

In [47]:
df['loan_to_od_avg'] = df['loan_amount_last'] / df['od_limit_avg']
df['od_range'] = df['overdraft_limit_max'] - df['overdraft_limit_min']
df['od_range_ratio'] = df['od_range'] / df['od_limit_avg']

In [48]:
df['weighted_spread'] = df['rate_spread'] * df['loan_amount_last']
df['stress_index'] = (df['loan_amount_last'] / df['overdraft_limit_max']) * df['offered_rate']
df['is_large_high_rate'] = (
    (df['loan_amount_last'] > df['loan_amount_last'].median()) &
    (df['rate_spread'] > df['rate_spread'].median())
).astype(int)
df['stress_index_is_inf'] = np.isinf(df['stress_index']).astype(int)
median_stress = df.loc[~np.isinf(df['stress_index']), 'stress_index'].median()
df['stress_index'].replace([np.inf, -np.inf], median_stress, inplace=True)
df['stress_index'].fillna(median_stress, inplace=True)

In [49]:
df['decision_date'] = pd.to_datetime(df['decision_day'])
df['weekday'] = df['decision_date'].dt.weekday
df['month'] = df['decision_date'].dt.month
df['quarter'] = df['decision_date'].dt.quarter
df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

In [50]:
region_counts = df['fl_adminarea'].value_counts()
df['region_freq'] = df['fl_adminarea'].map(region_counts)
product_counts = df['db_group_last'].value_counts()
df['last_product_freq'] = df['db_group_last'].map(product_counts)

threshold = 0.01 * len(df)
for col in ['fl_adminarea', 'db_group_last']:
    rare = df[col].value_counts()[df[col].value_counts() < threshold].index
    df[col] = df[col].apply(lambda x: 'Other' if x in rare else x)

In [51]:
y=df['target_value']
df.drop(columns=['front_id','decision_day',
                                    'corp_credit_products','sum_deb_ul_90',
                                    'sum_deb_ul_30','cnt_deb_loan_90',
                                    'sum_deb_ul_90','cnt_deb_ul_ip_30',
                                    'loan_rev_max_start_non_fin','loan_rev_min_start_fin',
                                    'app_term_mean_360','overdraft_app_term_max_360','days_from_authperson_registration',
                                    'corp_list','target_value','weekday',
                                    'p75_time_spent_minutes','sum_deb_investment_90',
                                    'cnt_deb_ul_ip_90','balance_rur_amt_30_min',
                                    'cnt_cred_loan_90','fl_hdb_bki_total_active_products','count_all_corp_dashboard_events'],axis=1,inplace=True)

In [52]:
df['db_group_last'].fillna('unknown',inplace=True)
df['fl_adminarea'].fillna('unknown',inplace=True)

In [53]:
df['region_freq'].fillna(df['region_freq'].median(),inplace=True)
df['last_product_freq'].fillna(df['last_product_freq'].median(),inplace=True)

In [54]:
df.isnull().sum()

loan_amount_last         0
overdraft_limit_min      0
overdraft_limit_max      0
offered_rate             0
cb_rate                  0
db_group_last            0
fl_adminarea             0
rate_spread              0
rate_ratio               0
log_rate_spread          0
od_limit_avg             0
loan_to_od_max           0
loan_to_od_avg           0
loan_to_od_max_is_inf    0
od_range                 0
od_range_ratio           0
weighted_spread          0
stress_index             0
is_large_high_rate       0
stress_index_is_inf      0
decision_date            0
month                    0
quarter                  0
weekday_sin              0
weekday_cos              0
region_freq              0
last_product_freq        0
dtype: int64

In [55]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [57]:
cat_features = ['db_group_last', 'fl_adminarea']
df_feature = df.copy()
numeric_features = [
    'loan_amount_last',
    'overdraft_limit_min',
    'overdraft_limit_max',
    'offered_rate',
    'cb_rate',
    'rate_spread',
    'rate_ratio',
    'log_rate_spread',
    'od_limit_avg',
    'loan_to_od_max',
    'loan_to_od_avg',
    'loan_to_od_max_is_inf',
    'od_range',
    'od_range_ratio',
    'weighted_spread',
    'stress_index',
    'is_large_high_rate',
    'stress_index_is_inf',
    'month',
    'quarter',
    'weekday_sin',
    'weekday_cos',
    'region_freq',
    'last_product_freq'
]


X = df_feature[numeric_features + cat_features]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    
])
pipe = Pipeline([
    ('prep', preprocess),
    ('model', GradientBoostingClassifier(subsample=1.0,n_estimators=300,
                                         min_samples_leaf=1,
                                         max_features='log2',
                                         min_samples_split=5,
                                         max_depth=4,
                                         learning_rate=0.05))
])


param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 4, 5],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 5],
    'model__max_features': ['sqrt', 'log2', None]
}
grid = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=3,
    verbose=2
)


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    pipe,
    param_distributions=param_grid,
    n_iter=30,
    cv=cv,
    scoring='roc_auc',
    n_jobs=4,
    random_state=42,
    verbose=2
)

pipe.fit(X, y)

# print("Лучшие параметры:", search.best_params_)
# print("Лучший ROC-AUC (CV):", search.best_score_)

# Оценка на всей выборке (осторожно, возможен оверфит)
y_pred = pipe.predict_proba(X)[:, 1]
print("ROC-AUC на обучении:", roc_auc_score(y, y_pred))

ROC-AUC на обучении: 0.7834064301694142
